# Catching a Stolen Credit Card with Isolation Forest

**One cardholder in Blacksburg, VA. 1,022 purchases. No fraud labels.** Can a model point us to the charges worth a phone call?



## Setup

In [ ]:
# ---- Setup: imports, styling, helper functions (run before the talk) ----
import warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from IPython.display import display
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # hide font warnings
warnings.filterwarnings("ignore")

# Slide colors + big fonts so plots are readable over Zoom
BG, NAVY, STEEL, LIGHT, RED, PEACH, MINT, DOT = "#FEF7EB", "#2B4D75", "#4E7DA2", "#A8D5E8", "#C8433A", "#F8E6CE", "#D8E8E7", "#3A78A6"
plt.rcParams.update({
    "figure.figsize": (12, 6.8), "figure.dpi": 100, "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "font.family": ["Lexend", "DejaVu Sans"], "font.size": 18, "axes.titlesize": 22, "axes.labelsize": 20,
    "xtick.labelsize": 17, "ytick.labelsize": 17, "legend.fontsize": 17, "text.color": NAVY,
    "axes.labelcolor": NAVY, "xtick.color": NAVY, "ytick.color": NAVY, "axes.edgecolor": STEEL,
    "lines.linewidth": 3, "axes.linewidth": 1.8, "axes.spines.top": False, "axes.spines.right": False,
})

# The two features the model sees
X_COL, Y_COL = "Amount", "MilesFromHome"
X_LABEL, Y_LABEL = "Purchase amount ($)", "Miles from home"
FULL = (-30, 1340, -40, 1900)       # full view, out to Phoenix
ZOOM = (0, 320, 0, 160)             # zoomed in on everyday purchases

def big_table(df, fmt=None):
    """Show a table in large font for Zoom."""
    display(df.style.format(fmt or {}).hide(axis="index").set_table_styles([
        {"selector": "th, td", "props": [("font-size", "20px"), ("padding", "6px 14px")]}]))

def style_axes(ax, lims=FULL):
    """Same labels, limits and $ formatting on every plot."""
    ax.set_xlabel(X_LABEL); ax.set_ylabel(Y_LABEL)
    ax.set_xlim(lims[0], lims[1]); ax.set_ylim(lims[2], lims[3])
    ax.xaxis.set_major_formatter(lambda v, _: f"${v:,.0f}"); ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

def isolate(X, target, rng, max_cuts=60):
    """Grow one random tree by hand (same rules as the model) and return the cuts it takes to isolate `target`.
    Simplified: uses all the data, no height limit, follows only the target's branch."""
    idx = np.arange(len(X)); lo, hi = X.min(0).astype(float), X.max(0).astype(float); cuts = []
    while len(idx) > 1 and len(cuts) < max_cuts:
        spread = X[idx].max(0) - X[idx].min(0)
        if not (spread > 0).any(): break
        f = rng.choice(np.flatnonzero(spread > 0))                  # random feature
        s = rng.uniform(X[idx, f].min(), X[idx, f].max())           # random cut between min and max
        cuts.append((f, s, lo.copy(), hi.copy()))
        if X[target, f] < s: idx = idx[X[idx, f] < s]; hi[f] = s    # keep the side the target is on
        else:                idx = idx[X[idx, f] >= s]; lo[f] = s
    return cuts

def draw_cuts(ax, X, target, cuts, title, color, lims=FULL):
    """Draw each cut as a dashed line across the box it split; title shows the cut count."""
    ax.scatter(X[:, 0], X[:, 1], s=60, c=LIGHT, edgecolor=STEEL, lw=0.8, alpha=0.6)
    for f, s, lo, hi in cuts:
        lo0, hi0 = max(lo[0], lims[0]), min(hi[0], lims[1]); lo1, hi1 = max(lo[1], lims[2]), min(hi[1], lims[3])
        if f == 0: ax.plot([s, s], [lo1, hi1], c=NAVY, lw=3, ls=(0, (5, 3)))     # amount cut: vertical
        else:      ax.plot([lo0, hi0], [s, s], c=NAVY, lw=3, ls=(0, (5, 3)))     # miles cut: horizontal
    ax.scatter(*X[target], s=450, c=color, edgecolor="white", lw=2.5, zorder=5)
    ax.set_title(f"{title}\nalone after {len(cuts)} cuts", fontweight="bold"); style_axes(ax, lims)

print("Ready.")

## Step 1: Load the purchases

 One person's credit card. They live in Blacksburg, and this is about a thousand of their purchases.

In [ ]:
df = pd.read_csv("card_transactions.csv")
print(f"{len(df):,} purchases. Typical: ${df.Amount.median():.0f}, {df.MilesFromHome.median():.0f} miles from home.")
big_table(df.head(), {"Amount": "${:,.2f}", "MilesFromHome": "{:,.1f}"})
## put this in slides

## Step 2: Plot every purchase

Two features: how much was spent, and how far from home

In [ ]:
X = df[[X_COL, Y_COL]].to_numpy()
fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], s=70, c=LIGHT, edgecolor=STEEL, lw=0.8, alpha=0.7)
style_axes(ax); ax.set_title(f"{len(df):,} purchases on one card", fontweight="bold")
plt.show()

## Step 3: Zoom into the corner

Zoomed in on bototm corner 

In [ ]:
fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], s=70, c=LIGHT, edgecolor=STEEL, lw=0.8, alpha=0.7)
style_axes(ax, ZOOM); ax.set_title("Zoomed in, closer to home", fontweight="bold")
plt.show()

## Step 4: Pick two purchases to compare



In [ ]:
odd = df.index[df.City == "Phoenix, AZ"][0]
normal = df.index[df.TransactionID == "T0108"][0]
big_table(df.loc[[odd, normal]], {"Amount": "${:,.2f}", "MilesFromHome": "{:,.1f}"})

## Step 5: Grow one random tree

Same as slide with the random cuts. Phoenix is alone after just 2 cuts. The grocery run takes 17, because it's buried in the crowd. The right plot is zoomed in.

In [ ]:
rng = np.random.default_rng(15)    # one tree whose cut counts match the average over 1,000 trees
fig, axes = plt.subplots(1, 2, figsize=(16, 6.8))
draw_cuts(axes[0], X, odd,    isolate(X, odd, rng),    "$1,284 in Phoenix", RED)
draw_cuts(axes[1], X, normal, isolate(X, normal, rng), "$67 groceries (zoomed in)", DOT, lims=ZOOM)
plt.tight_layout(); plt.show()

## Step 6: Build the forest

Now the real thing, in scikit-learn. It builds 500 trees, each from a random sample of 256 purchases.

In [ ]:
model = IsolationForest(n_estimators=100, random_state=0)
model.fit(df[[X_COL, Y_COL]])
print(f"Built {len(model.estimators_)} trees, each from a random sample of {model.max_samples_} purchases.")

## Step 7: Score every purchase

> **Say:** Then every purchase is dropped down every tree. Lower score means it was isolated faster, so it's more unusual.

In [ ]:
df["score"] = model.score_samples(df[[X_COL, Y_COL]])   # scikit-learn's version: lower = more unusual
df["s"] = -df["score"]                                   # flipped: higher = more unusual (matches our slides)
big_table(df[["TransactionID", "Merchant", "City", "Amount", "MilesFromHome", "s"]].head(),
          {"Amount": "${:,.2f}", "MilesFromHome": "{:,.1f}", "s": "{:.3f}"})

## Step 8: Look at all the scores

> **Say:** Most purchases score about the same. There's a tail off to the left, but no clean break between weird and normal. scikit-learn shows the negative of s, so −0.8 means s ≈ 0.8: well above the 0.5 line from our scoring slide.

In [ ]:
df["s"] = -df.score                                  # scikit-learn reports -s; flip it so higher = more unusual
jitter = np.random.default_rng(0).uniform(-1, 1, len(df))
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.scatter(df.s, jitter, s=55, c=LIGHT, edgecolor=STEEL, lw=0.8, alpha=0.75)
ax.axvline(0.5, color=STEEL, lw=2, ls=":"); ax.text(0.5, 1.25, "0.5", ha="center", fontsize=17, color=STEEL)
ax.set_yticks([]); ax.spines["left"].set_visible(False); ax.set_ylim(-1.4, 1.5); ax.set_xlim(0.33, 0.9)
ax.set_xlabel("score s  (higher = more unusual)"); ax.set_title("One dot per purchase", fontweight="bold")
plt.show()

## Step 9: Pick a threshold

> **Say:** The model ranks; we decide where to cut. scikit-learn's default flags 130 purchases, way too many phone calls. With no clean break, the real question is how many calls the bank can make. Say 10.

In [ ]:
CUT_S = 0.68                                         # flag anything with s above this
df["flagged"] = df.s > CUT_S
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.scatter(df.s[~df.flagged], jitter[~df.flagged], s=55, c=LIGHT, edgecolor=STEEL, lw=0.8, alpha=0.75)
ax.scatter(df.s[df.flagged], jitter[df.flagged], s=170, c=RED, edgecolor="white", lw=1.5, zorder=5)
ax.axvline(0.5, color=STEEL, lw=2, ls=":"); ax.text(0.5, 1.25, "default", ha="center", fontsize=17, color=STEEL)
ax.axvline(CUT_S, color=NAVY, lw=3, ls="--"); ax.text(CUT_S, 1.25, "our cut", ha="center", fontsize=17, color=NAVY, fontweight="bold")
ax.set_yticks([]); ax.spines["left"].set_visible(False); ax.set_ylim(-1.4, 1.5); ax.set_xlim(0.33, 0.9)
ax.set_xlabel("score s  (higher = more unusual)"); ax.set_title(f"Our cut: s > {CUT_S}  ({df.flagged.sum()} flagged, red)", fontweight="bold")
plt.show()
print(f"scikit-learn's default threshold would flag {(model.predict(df[[X_COL, Y_COL]]) == -1).sum()} purchases. Our cut flags {df.flagged.sum()}.")

## Step 10: See what got flagged

> **Say:** Here's the model's map. Darker red is more unusual, and the red dots are our 10. They're not all the same kind of weird.

In [ ]:
gx, gy = np.meshgrid(np.linspace(*FULL[:2], 300), np.linspace(*FULL[2:], 300))
gz = -model.score_samples(pd.DataFrame({X_COL: gx.ravel(), Y_COL: gy.ravel()})).reshape(gx.shape)   # s: higher = more unusual
flag = df[df.flagged]

fig, ax = plt.subplots(figsize=(11, 6))
shade = ax.contourf(gx, gy, gz, levels=20, cmap="Reds", alpha=0.5)
ax.scatter(X[:, 0], X[:, 1], s=45, c=NAVY, alpha=0.45, lw=0)
ax.scatter(flag[X_COL], flag[Y_COL], s=280, c=RED, edgecolor="white", lw=2.5, zorder=5)
cbar = fig.colorbar(shade, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("score s (darker = more unusual)", fontsize=14)
cbar.set_ticks([0.4, 0.5, 0.6, 0.7, 0.8])
cbar.ax.tick_params(labelsize=12)
cbar.outline.set_visible(False)

style_axes(ax); ax.set_title("Purchases worth a further look", fontweight="bold")
plt.tight_layout(); plt.show()

## Step 11: Why does each one look weird?

> **Say:** Two kinds of weird. Big purchases far from home, in eight different cities. And big purchases right at home.

In [ ]:
short = df.nsmallest(12, "score").copy()
short.insert(0, "rank", range(1, 13))
big_table(short[["rank", "Merchant", "City", "Amount", "MilesFromHome"]],
          {"Amount": "${:,.2f}", "MilesFromHome": "{:,.0f}"})

## Step 12: Weird is not the same as bad

> **Say:** The model can't tell us which ones are fraud just which ones are weird. Up to the an analyst at the CC company to decide to make a call

In [ ]:
df["rank"] = df.score.rank().astype(int)
pair = df[(df.Merchant == "Hotel") | (df.City == "Richmond, VA")].sort_values("rank").copy()
pair["purchases in this city"] = pair.City.map(df.City.value_counts())
big_table(pair[["rank", "Merchant", "City", "Amount", "purchases in this city"]], {"Amount": "${:,.2f}"})